# Confirmatory Analysis 06 — Seasonality

**Goal:** Confirm whether MTG prices exhibit seasonal patterns tied to the set release cycle (~90 days). This CDA extends Stat 03 (seasonal_decomposition.ipynb) to a confirmatory framework.

**Tables:** gold_price_features (daily market medians)

**α = 0.05**

---
## Hypotheses
1. Market-wide median price follows a seasonal pattern with period ≈ 90 days (new set release cycle) — STL decomposition confirms a non-negligible seasonal component
2. Tier 1 (<€100) prices show stronger seasonal amplitude than Tier 3 (>€1000) — new set releases depress bulk prices more than premium/Reserved List prices

⚠️ **Data requirement:** H1 requires STL with period=7 (weekly proxy): n ≥ 15 days. For the proper quarterly period=90: n ≥ 180 days. Both are currently deferred.

In [1]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau
from statsmodels.tsa.seasonal import STL

In [2]:
gold = duckdb.connect("../../data/gold/cards.duckdb", read_only=True)

In [3]:
# PERCENTILE_CONT(0.5) is SQL median — used instead of AVG because the price
# distribution is Pareto (α=1.303), making the mean heavily skewed by Power Nine outliers
market = gold.execute("""
    SELECT snapshot_date,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY eur) AS median_eur
    FROM gold_price_features
    WHERE eur IS NOT NULL
    GROUP BY snapshot_date
    ORDER BY snapshot_date
""").df()
market["snapshot_date"] = pd.to_datetime(market["snapshot_date"])
market = market.set_index("snapshot_date")

# log1p transformation stabilises variance before decomposition — STL assumes
# an approximately additive model, which holds better in log space than in EUR space
market_log = np.log1p(market["median_eur"])

n = len(market)
# These thresholds come from STL requirements: period must be ≤ n/2,
# and period=7 (weekly) needs at least 2 full cycles = 14 days → use 15 as safe minimum
min_n_weekly = 15
min_n_quarterly = 180

print(
    f"Days available: {n}  (need {min_n_weekly} for period=7; {min_n_quarterly} for period=90)"
)
if n < min_n_weekly:
    print("⚠ INSUFFICIENT DATA — all tests deferred.")
    print(
        f"  STL weekly (period=7):    re-run at ≥{min_n_weekly} days (~{market.index.max().date() + pd.Timedelta(days=min_n_weekly - n)})"
    )
    print(f"  STL quarterly (period=90): re-run at ≥{min_n_quarterly} days")

Days available: 5  (need 15 for period=7; 180 for period=90)
⚠ INSUFFICIENT DATA — all tests deferred.
  STL weekly (period=7):    re-run at ≥15 days (~2026-06-18)
  STL quarterly (period=90): re-run at ≥180 days


## H1 — Seasonal STL Pattern (Mann-Kendall + STL)

**Hypothesis:** The STL seasonal component explains a non-trivial fraction of price variance.

**Test:** STL decomposition → seasonal amplitude / total variance. If seasonal amplitude / std(series) > 0.05 → seasonal effect is non-negligible.

**Mann-Kendall:** Tests for a monotonic trend. Non-significant trend = prices are flat or random-walked, not trending — relevant for feature engineering (lag features vs level features).

**Threshold:** seasonal amplitude / series std > 0.05 = CONFIRMED; < 0.01 = REJECTED.

In [4]:
if n < min_n_weekly:
    rerun = (market.index.max() + pd.Timedelta(days=min_n_weekly - n)).date()
    print(f"INSUFFICIENT DATA: {n} days (need ≥{min_n_weekly} for STL period=7).")
    print(f"Re-run after approximately {rerun}.")
    print()
    print("Once data is available:")
    print("  STL(market_log, period=7 or 90, robust=True).fit()")
    print(
        "  seasonal_fraction = (result.seasonal.max() - result.seasonal.min()) / market_log.std()"
    )
    print("  CONFIRMED if seasonal_fraction > 0.05")
else:
    # Use the quarterly period (90 days = MTG set release cycle) when we have enough data;
    # fall back to period=7 (weekly) as a proxy that can detect any cyclical pattern at all
    period = 90 if n >= min_n_quarterly else 7
    period_label = (
        "90 (quarterly MTG cycle)" if n >= min_n_quarterly else "7 (weekly proxy)"
    )
    print(f"Period used: {period_label}")

    # robust=True: down-weights price spike outliers (ban announcements, buyout events)
    # so the decomposition describes normal seasonal behaviour, not one-off shocks
    stl = STL(market_log, period=period, seasonal=7, robust=True)
    result = stl.fit()

    # Amplitude = peak-to-trough of the seasonal component in log-EUR units
    seasonal_amp = result.seasonal.max() - result.seasonal.min()
    # Normalise by series std to make the threshold (0.05) scale-independent;
    # a 5% fraction means seasonality explains at least 5% of total variation
    seasonal_fraction = seasonal_amp / market_log.std()
    trend_range = result.trend.max() - result.trend.min()
    resid_std = result.resid.std()
    confirmed = seasonal_fraction > 0.05

    print(f"Seasonal amplitude:           {seasonal_amp:.4f}")
    print(f"Seasonal fraction (amp/std):  {seasonal_fraction:.4f}")
    print(f"Trend range:                  {trend_range:.4f}")
    print(f"Residual std:                 {resid_std:.4f}")
    print(
        f"H1 result: {'CONFIRMED (seasonal_fraction > 0.05)' if confirmed else 'NOT CONFIRMED (< 0.05)'}"
    )

    # Mann-Kendall: non-parametric monotonic trend test — preferred over linear regression
    # because price series violate the normality assumption required for OLS significance tests
    t = np.arange(len(market_log))
    tau, p_mk = kendalltau(t, market_log.values)
    print(f"\nMann-Kendall τ={tau:.4f}, p={p_mk:.4e}")
    print(
        f"Trend: {'SIGNIFICANT' if p_mk < 0.05 else 'not significant'} "
        f"({'upward' if tau > 0 else 'downward'} if significant)"
    )

    # Four-panel plot: original / trend / seasonal / residuals
    # sharex=True links the date axis so zooming one panel zooms all four simultaneously
    fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)
    axes[0].plot(market_log.index, market_log, color="steelblue")
    axes[0].set_title("Original log1p(market median EUR)")
    axes[1].plot(market_log.index, result.trend, color="darkorange")
    axes[1].set_title("Trend")
    axes[2].plot(market_log.index, result.seasonal, color="green")
    axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[2].set_title("Seasonal")
    axes[3].plot(market_log.index, result.resid, color="gray")
    axes[3].axhline(0, color="red", linewidth=0.8, linestyle="--")
    axes[3].set_title("Residuals")
    plt.suptitle(f"STL Decomposition — period={period}", y=1.01, fontsize=12)
    plt.tight_layout()
    plt.show()

INSUFFICIENT DATA: 5 days (need ≥15 for STL period=7).
Re-run after approximately 2026-06-18.

Once data is available:
  STL(market_log, period=7 or 90, robust=True).fit()
  seasonal_fraction = (result.seasonal.max() - result.seasonal.min()) / market_log.std()
  CONFIRMED if seasonal_fraction > 0.05


## H2 — Tier 1 vs Tier 3 Seasonal Amplitude

**Hypothesis:** Tier 1 (<€100) cards show larger seasonal amplitude than Tier 3 (>€1000).

**Rationale:** New set releases depress bulk common/uncommon prices (new supply enters the market). Reserved List and Power Nine cards (Tier 3) are not reprinted — their prices depend on speculative events, not set releases.

**Test:** Run STL separately per tier. Compare seasonal amplitude (seasonal.max − seasonal.min) and seasonal fraction (amplitude / std).

In [5]:
if n < min_n_weekly:
    rerun = (market.index.max() + pd.Timedelta(days=min_n_weekly - n)).date()
    print(f"INSUFFICIENT DATA: {n} days (need ≥{min_n_weekly}). Re-run after {rerun}.")
    print()
    print("Expected result:")
    print("  Tier 1 seasonal fraction: HIGHER (bulk prices react strongly to new sets)")
    print("  Tier 3 seasonal fraction: LOWER  (RL/Power Nine prices are event-driven)")
else:
    period = 90 if n >= min_n_quarterly else 7
    tier_results = []
    # 2-row × 3-col grid: rows = tiers, columns = Trend / Seasonal / Residuals
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for row_idx, (label, lo, hi) in enumerate(
        [("Tier 1 (<€100)", 0, 100), ("Tier 3 (>€1000)", 1000, 1e9)]
    ):
        # f-string SQL: the tier bounds lo and hi are Python variables injected into the query;
        # safe here because both are numeric literals, not user-supplied strings
        ts = gold.execute(f"""
            SELECT snapshot_date,
                PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY eur) AS median_eur
            FROM gold_price_features
            WHERE eur > {lo} AND eur < {hi}
            GROUP BY snapshot_date ORDER BY snapshot_date
        """).df()
        ts["snapshot_date"] = pd.to_datetime(ts["snapshot_date"])
        ts = ts.set_index("snapshot_date")
        ts_log = np.log1p(ts["median_eur"])
        # STL requires at least 2 full periods of data; skip gracefully if not yet available
        if len(ts_log) < 2 * period + 1:
            for ax in axes[row_idx]:
                ax.text(
                    0.5,
                    0.5,
                    f"{label}\nInsufficient data",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
            tier_results.append(
                {"tier": label, "n": len(ts_log), "seasonal_fraction": None}
            )
            continue
        stl_t = STL(ts_log, period=period, robust=True).fit()
        amp = stl_t.seasonal.max() - stl_t.seasonal.min()
        frac = amp / ts_log.std()
        tier_results.append(
            {
                "tier": label,
                "n": len(ts_log),
                "seasonal_amplitude": round(amp, 4),
                "seasonal_fraction": round(frac, 4),
            }
        )
        # Plot each STL component in the correct row/column of the grid
        for ax, component, title, color in zip(
            axes[row_idx],
            [stl_t.trend, stl_t.seasonal, stl_t.resid],
            ["Trend", "Seasonal", "Residuals"],
            ["darkorange", "green", "gray"],
        ):
            ax.plot(ts_log.index, component, color=color)
            ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
            ax.set_title(f"{label} — {title}", fontsize=9)
    plt.tight_layout()
    plt.show()
    print(pd.DataFrame(tier_results).to_string(index=False))

INSUFFICIENT DATA: 5 days (need ≥15). Re-run after 2026-06-18.

Expected result:
  Tier 1 seasonal fraction: HIGHER (bulk prices react strongly to new sets)
  Tier 3 seasonal fraction: LOWER  (RL/Power Nine prices are event-driven)


In [6]:
gold.close()

## 📋 Final Conclusions

```
DATA
─────────────────────────────────────────────────────────────────────────────
Days of history: 3  (need ≥15 for STL period=7; ≥180 for period=90)
All tests: DEFERRED

H1 — Seasonal STL pattern:
  Seasonal fraction: N/A (STL not run)
  Mann-Kendall τ:    N/A
  DEFERRED — re-run after ≥15 days (~2026-06-19)

H2 — Tier 1 vs Tier 3 seasonal amplitude:
  Tier 1 amplitude: N/A
  Tier 3 amplitude: N/A
  DEFERRED — re-run after ≥15 days

MODEL IMPLICATIONS (once data available)
─────────────────────────────────────────────────────────────────────────────
If H1 CONFIRMED: add seasonal features to the model (day-of-quarter, set-release-offset)
If H2 CONFIRMED: use tier-specific seasonal features (bulk cards need it; Tier 3 doesn't)
If trend is significant (Mann-Kendall): model levels with trend term, not just returns

RETEST SCHEDULE
─────────────────────────────────────────────────────────────────────────────
STL weekly (period=7):     ≈ 2026-06-19  (≥15 daily snapshots)
STL quarterly (period=90): ≈ 2026-11-30  (≥180 daily snapshots)
```